# Trabalho Prático 1 - Aprendizagem Automática
## Previsão de Preços de Carros Usados (Kaggle Competition)

**Licenciatura em Engenharia de Sistemas e Tecnologias Informáticas** 

**Unidade Curricular:** Aprendizagem Automática  
**Ano Letivo:** 2025/2026 

---

### Identificação do Grupo
* **Aluno 1:** Bernardo Freitas (79295)
* **Docente:** Prof. Pedro Cardoso

---

O objetivo deste trabalho prático (Parte 1) é desenvolver e otimizar modelos de **Aprendizagem Supervisionada** capazes de prever o preço de venda de carros usados com base nas suas características físicas e mecânicas.

## 1. Import das Bibliotecas e dos Algoritmos

In [1]:
# --- 1. Ferramentas de Manipulação e Sistema ---
import os
import json
import numpy as np
import pandas as pd
from datetime import date

# --- 2. Ferramentas de Pré-processamento e Validação ---
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# --- 3. Algoritmos de Machine Learning ---
# Modelos Lineares
from sklearn.linear_model import LinearRegression, Ridge, Lasso
# Modelos baseados em Distância e Vetores
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
# Modelos baseados em Árvores (Ensembles)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
# Redes Neuronais
from sklearn.neural_network import MLPRegressor

## 2. Feature Engineering

In [2]:
def process_data_features(df_input):
    # Criar cópia para processamento
    data = df_input.copy()

    # 1. Limpeza Básica
    data = data.drop_duplicates()
    
    # 2. Extração de Dados do Motor
    data['hp'] = data['engine'].str.extract(r'(\d+\.?\d*)HP', expand=False).astype(float)
    data['liters'] = data['engine'].str.extract(r'(\d+\.?\d*)L\s', expand=False).astype(float)
    
    # Lógica de cilindros: tenta primeiro "X Cylinder", se falhar tenta "VX"
    cyl_text = data['engine'].str.extract(r'(\d+)\s+Cylinder', expand=False)
    cyl_v = data['engine'].str.extract(r'V(\d+)', expand=False)
    data['cylinders'] = cyl_text.fillna(cyl_v).astype(float)

    # 3. Features Temporais (Idade do Carro)
    current_year = date.today().year
    data['car_age'] = current_year - data['model_year']
    data['car_age'] = data['car_age'].replace(0, 1) 

    # 4. Detalhes Técnicos do Motor
    data['is_turbo'] = data['engine'].str.contains(r'(?i)turbo', na=False).astype(int)
    data['turbo_type'] = data['engine'].str.extract(r'(Twin Turbo|Turbo)', expand=False)
    data['valve_train'] = data['engine'].str.extract(r'(DOHC|OHV|SOHC)', expand=False) 
    data['fuel_injection'] = data['engine'].str.extract(r'(PDI|GDI|MPFI)', expand=False)

    # 5. Engenharia de Uso (Miles per Year)
    data['miles_p_year'] = data['milage'] / data['car_age']

    # 6. Padronização de Categorias (Combustível e Transmissão)
    data['fuel_type'] = data['fuel_type'].apply(lambda x: 
        'Hybrid' if 'hybrid' in str(x).lower() else 
        ('EV' if 'not supported' in str(x).lower() else x)
    )

    def classify_trans(val):
        v = str(val).lower()
        if any(x in v for x in ['automatic', 'a/t', 'cvt']): return 'Automatico'
        if any(x in v for x in ['manual', 'm/t']): return 'Manual'
        return 'Outro'
    data['transmission_type'] = data['transmission'].apply(classify_trans)

    # 7. Redução de Cardinalidade (Cores)
    for col, new_col in [('ext_col', 'ext_col_simple'), ('int_col', 'int_col_simple')]:
        top_10 = data[col].value_counts().nlargest(10).index
        data[new_col] = data[col].apply(lambda x: x if x in top_10 else 'Other')

    # 8. Tratamento de Nulos e Strings
    obj_cols = data.select_dtypes(include=['object']).columns
    data[obj_cols] = data[obj_cols].replace('-', 'Unknown').fillna('Unknown')
    data['clean_title'] = data['clean_title'].replace('Unknown', 'No')

    # 9. Variáveis Finais e Transformações
    data['accident_clean'] = data['accident'].apply(lambda x: 0 if 'None' in str(x) else 1)
    
    # Feature Interaction (Ratios)
    data['hp_per_liter'] = data['hp'] / (data['liters'] + 0.001)
    data['hp_per_cylinder'] = data['hp'] / (data['cylinders'] + 0.001)
    
    # Transformação Logarítmica (Target/Features com alta variância)
    data['milage_log'] = np.log1p(data['milage'])

    return data

## 3. Preparação e Transformação de Dados (Data Preparation)

Nesta etapa, consolido o pipeline de processamento convertendo os dados limpos em matrizes numéricas prontas para serem consumidas pelos algoritmos de Machine Learning.

In [3]:
def prepare_model_inputs(df_train, df_test):
    """
    Preparação final: separa Target, preenche nulos e codifica texto.
    """
    # 1. Aplicar a Engenharia de Features criada anteriormente
    train_proc = process_data_features(df_train)
    test_proc = process_data_features(df_test)

    # 2. Definição de Variáveis (Seleção de Features)
    NUM_FEATURES = [
        'hp', 'liters', 'car_age', 'cylinders', 'miles_p_year', 
        'milage', 'model_year', 'is_turbo'
    ]
    # Texto
    CAT_FEATURES = [
        'brand', 'model', 'fuel_type', 'transmission_type', 
        'ext_col_simple', 'int_col_simple', 'clean_title', 
        'turbo_type', 'valve_train', 'fuel_injection'
    ]

    # 3. Processamento Numérico (Imputação Simples)
    # Preencho vazios com 0, mais seguro que inventar uma media. 
    X_train_num = train_proc[NUM_FEATURES].fillna(0)
    X_test_num = test_proc[NUM_FEATURES].fillna(0)

    # 4. Codificação de Variáveis Categóricas (Label Encoding)
    X_train_cat = train_proc[CAT_FEATURES].astype(str).copy()
    X_test_cat = test_proc[CAT_FEATURES].astype(str).copy()

    for col in CAT_FEATURES:
        le = LabelEncoder()
        
        # Treinar o codificador apenas com os dados de Treino
        X_train_cat[col] = le.fit_transform(X_train_cat[col])

        # {Categoria: Numero}
        le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
        
        # Uso .map(): Se o valor existir no dicionário, converte. 
        # Se não existir (carro novo no teste), fica NaN e depois preencho com -1.
        X_test_cat[col] = X_test_cat[col].map(le_dict).fillna(-1).astype(int)

    # 5. Consolidação Final
    # Separar o Target (y, o preco) 
    y_train = train_proc['price']
    
    # Juntar colunas numéricas e categóricas tratadas (x, idade, marca, etc)
    X_train_final = pd.concat([X_train_num, X_train_cat], axis=1)
    X_test_final = pd.concat([X_test_num, X_test_cat], axis=1)

    return X_train_final, y_train, X_test_final

## 4. Definição do Modelo Preditivo

Nesta etapa, procedo à inicialização do algoritmo de Machine Learning selecionado para o problema de regressão.

Optei pela utilização exclusiva do **XGBoost**. Esta escolha deve-se à natureza tabular do dataset, onde algoritmos de *boosting* (que constroem árvores de decisão sequencialmente para corrigir erros anteriores) demonstram consistentemente performance superior a modelos lineares ou redes neuronais simples.

In [4]:
def initialize_regressors():
    """
    Inicializa o modelo selecionado para o treino, focado no XGBoost.
    """
    # Dicionário de algoritmos
    regressors = {
        # random_state=42: Garante que os resultados são sempre iguais (reprodutibilidade)
        'XGBoost_Main': xgb.XGBRegressor(random_state=42, n_jobs=-1)
    }

    return regressors

## 5. GridSearch com Cross Validation

Para maximizar a capacidade preditiva do XGBoost, não é suficiente utilizar as configurações padrão. É necessário encontrar a combinação ótima de parâmetros que equilibre a capacidade de aprendizagem com a generalização.

Utilizei a técnica de **Grid Search com Cross-Validation**, implementada através da função `tune_model_hyperparameters`.

In [5]:
def get_hyperparameter_space(model_name):
    """
    Devolve o conjunto de todas as combinações possíveis de parâmetros que definimos para o modelo testar, para a otimização dos algoritmos.
    Focado na otimização do XGBoost para evitar overfitting e melhorar a generalização.
    """

    if 'XGBoost' in model_name:
        return {
            # Quantidade de árvores (mais árvores = mais aprendizagem, mas mais lento)
            'n_estimators': [100, 300, 500],
            
            # Velocidade de aprendizagem (menor = mais preciso, exige mais árvores)
            'learning_rate': [0.05, 0.1],
            
            # Profundidade da árvore (maior = mais complexo)
            'max_depth': [4, 6, 8],

            'subsample': [0.8],
            'colsample_bytree': [0.8],
            
            # Regularização onde Alpha (L1) e Lambda (L2) ajudam a "cortar" features inúteis
            'reg_alpha': [0, 0.1],
            'reg_lambda': [1, 1.5]
        }
    
    # Caso use outro modelo no futuro
    return {}

def tune_model_hyperparameters(model, model_name, X_train, y_train):
    """
    Executa o GridSearch com o Cross-Validation.
    """
    print(f"\nA otimizar hiperparâmetros para: {model_name}...")
    
    param_grid = get_hyperparameter_space(model_name)
    
    # Se não houver grelha definida, devolve o modelo original sem perder tempo
    if not param_grid:
        print(f"   -> Nenhuma grelha encontrada. A usar parâmetros default.")
        return model

    # Configuração do GridSearch (tecnica de otimização de hiperparâmetros)
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5, # Treina 5 vezes, 4 para treino e 1 para validação
        scoring='neg_root_mean_squared_error', # Usei RMSE negativo porque o GridSearch maximiza a métrica
        verbose=1,
        n_jobs=-1 
    )

    # Executar o treino 
    grid_search.fit(X_train, y_train)

    # Reportar resultados
    print(f"Otimização Concluída!")
    print(f"Melhor RMSE (Média CV): {-grid_search.best_score_:.4f}")
    print(f"Melhores Parâmetros: {grid_search.best_params_}")
    
    return grid_search.best_estimator_

## 6. Avaliação e Diagnóstico de Performance

Nesta etapa, vamos ver se o computador acertou no preço ou se errou muito. Vou usar as métricas (MAE, RMSE e o R²) para medir esse erro. A função `evaluate_model_performance` calcula e compara o desempenho nos conjuntos de Treino e Validação.

In [6]:
def evaluate_model_performance(model, X_train, y_train, X_val, y_val, model_name):
    """
    Treina o modelo, gera previsões e calcula métricas detalhadas de desempenho.
    """
    print(f"\n{'='*80}")
    print(f"A Avaliar: {model_name}")
    print(f"{'='*80}")

    # 1. Treino 
    model.fit(X_train, y_train)

    # 2. Previsões
    pred_train = model.predict(X_train) # Tenta adivinhar adivinhar o preço dos carros que já viu (Treino)
    pred_val = model.predict(X_val) # Tenta adivinhar o preço dos carros que nunca viu (Validação)

    # 3. Função interna para calcular métricas
    def get_metrics(y_true, y_pred):
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        return rmse, mae, r2

    # Calcular métricas
    rmse_train, mae_train, r2_train = get_metrics(y_train, pred_train) # Nota: Treino
    rmse_val, mae_val, r2_val = get_metrics(y_val, pred_val) # Nota: Final

    # 4. Exibição Profissional
    print(f"PERFORMANCE:")
    print(f"{'Métrica':<10} | {'Treino':>12} | {'Validação':>12} | {'Diferença (Gap)':>15}")
    print(f"{'-'*60}")
    
    # RMSE (Erro Quadrático Médio)
    gap_rmse = rmse_val - rmse_train 
    print(f"{'RMSE':<10} | {rmse_train:12,.2f} | {rmse_val:12,.2f} | {gap_rmse:+15,.2f}")
    
    # MAE (Erro Absoluto Médio)
    print(f"{'MAE':<10} | {mae_train:12,.2f} | {mae_val:12,.2f} |")
    
    # R2 (Coeficiente de Determinação)
    print(f"{'R²':<10} | {r2_train:12.4f} | {r2_val:12.4f} |")

    # 5. Diagnóstico Rápido
    if r2_val < 0.5:
        print(f"\nALERTA: O modelo tem baixa performance (R² baixo). Possível Underfitting.")
    elif (rmse_val - rmse_train) > 20000: # Valor arbitrário de exemplo
        print(f"\nALERTA: Grande diferença entre Treino e Validação. Possível Overfitting.")

    return {
        'model_name': model_name,
        'model_obj': model,
        'rmse_val': rmse_val,
        'r2_val': r2_val,
        'mae_val': mae_val
    }

## 7. Treino

A função `run_training_pipeline` atua como o orquestrador central do projeto, integrando todas as etapas anteriores num fluxo contínuo.

In [7]:
def run_training_pipeline(df_raw_train, df_raw_test):
    """
    Orquestrador Principal:
    1. Prepara os dados
    2. Divide em Treino/Validação
    3. Executa a Otimização (GridSearch)
    4. Avalia e seleciona o melhor modelo
    """
    
    # 1. Preparação dos Dados (Chama a função da Etapa 3)
    print("A processar e transformar dados")
    X, y, X_test_submission = prepare_model_inputs(df_raw_train, df_raw_test)

    # 2. Split Treino/Validação (80% Treino, 20% Validação)
    # random_state=42 garante que a divisão é sempre igual
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    print(f"Setup Concluído:")
    print(f"   - Treino: {X_train.shape[0]} carros")
    print(f"   - Validação: {X_val.shape[0]} carros")
    print(f"   - Features: {X_train.shape[1]} colunas")

    # 3. Carregar Dicionário de Modelos (Chama a função da Etapa 4)
    models_map = initialize_regressors()
    
    results_list = []
    trained_models_dict = {}

    # 4. Loop de Treino e Avaliação
    for name, model in models_map.items():
        
        # A. Otimização de Hiperparâmetros (GridSearch - Etapa 5)
        best_model_tuned = tune_model_hyperparameters(model, name, X_train, y_train)

        # B. Avaliação de Performance (Etapa 6)
        metrics = evaluate_model_performance(
            best_model_tuned, 
            X_train, y_train, 
            X_val, y_val, 
            name
        )

        # C. Guardar resultados
        results_list.append(metrics)
        trained_models_dict[name] = best_model_tuned

    # 5. Consolidação e Escolha do Vencedor
    df_results = pd.DataFrame(results_list)
    
    # Ordenar pelo menor erro de validação (RMSE)
    df_results = df_results.sort_values(by='rmse_val', ascending=True)
    
    print(f"\n\n{'='*60}")
    print("CLASSIFICAÇÃO FINAL (Leaderboard)")
    print(f"{'='*60}")
    # Mostrar apenas colunas relevantes
    display_cols = ['model_name', 'rmse_val', 'r2_val', 'mae_val']
    print(df_results[display_cols].to_string(index=False))

    return df_results, trained_models_dict, X_test_submission

## 8. Versionamento e Exportação de Resultados

Nesta etapa final, persisti os resultados gerados modelo. 

In [8]:
def save_project_artifacts(df_submission, model_obj, metrics_dict):
    """
    Guarda o CSV para submissão no Kaggle e gera um Log (JSON) com os detalhes técnicos.
    Utiliza um sistema de ID incremental (1, 2, 3...) para manter histórico de versões.
    """
    
    # 1. Configurar pasta de destino 
    output_dir = 'sub'
    os.makedirs(output_dir, exist_ok=True) 

    # 2. Lógica de Auto-Incremento 
    # Lê os ficheiros na pasta para saber se estamos na submissão 1, 5 ou 10.
    existing_files = os.listdir(output_dir)
    existing_ids = []
    
    for f in existing_files:
        if f.startswith('submission_') and f.endswith('.csv'):
            try:
                # Exemplo: 'submission_4.csv' -> extrai o 4
                id_num = int(f.replace('submission_', '').replace('.csv', ''))
                existing_ids.append(id_num)
            except ValueError:
                continue
    
    # Se a lista estiver vazia começa no 1, senão soma 1 ao maior que encontrou
    next_id = max(existing_ids) + 1 if existing_ids else 1

    # 3. Definir caminhos dos ficheiros
    filename_csv = f"{output_dir}/submission_{next_id}.csv"
    filename_json = f"{output_dir}/submission_{next_id}_log.json"

    # 4. Guardar o CSV (Formato exigido pelo Kaggle)
    # index=False é crucial para não criar uma coluna extra de índices (0, 1, 2...)
    df_submission.to_csv(filename_csv, index=False)

    # 5. Extrair e Guardar Metadados (O "Segredo" da Defesa)
    try:
        # Tenta extrair a configuração exata do XGBoost
        params = model_obj.get_params()
    except:
        params = "Configuração não disponível"

    log_data = {
        "id": next_id,
        "model_type": "XGBoost",
        "validation_metrics": metrics_dict,  # O RMSE que obtiveste
        "hyperparameters": params            # A configuração usada (n_estimators, depth, etc.)
    }

    # Gravar o ficheiro de Log (JSON)
    with open(filename_json, 'w', encoding='utf-8') as f:
        json.dump(log_data, f, indent=4, default=str)

    print(f"\nSUBMISSÃO #{next_id} FEITA")
    print(f"Ficheiro CSV: {filename_csv} (Enviar para o Kaggle)")
    print(f"Log Técnico:  {filename_json} (Para referência futura)")
    
    return filename_csv

# Botão ON

In [ ]:
if __name__ == "__main__":
    
    print("INICIAR O PROJETO...")
    
    # 1. Carregar os Dados (Ajusta o caminho se necessário)
    # index_col='id' usa a coluna 'id' como índice, facilitando a submissão
    try:
        df_train_raw = pd.read_csv('train.csv', index_col='id')
        df_test_raw = pd.read_csv('test.csv', index_col='id')
        print("Dados carregados com sucesso. (Na raiz)")
    except FileNotFoundError:
        print("Erro: Ficheiros 'train.csv' e 'test.csv' não encontrados.")
        # fallback para caminhos alternativos, ex: data/train.csv
        df_train_raw = pd.read_csv('data/train.csv', index_col='id')
        df_test_raw = pd.read_csv('data/test.csv', index_col='id')

    # 2. Executar o Pipeline Completo (Chama a função da Etapa 7)
    # Isto vai limpar, treinar, otimizar e avaliar tudo sozinho.
    df_leaderboard, models_dict, X_test_final = run_training_pipeline(df_train_raw, df_test_raw)

    # 3. Selecionar o champion
    # Pega na primeira linha da tabela de resultados (o que teve menor erro)
    best_run = df_leaderboard.iloc[0]
    best_model_name = best_run['model_name']
    best_model_obj = models_dict[best_model_name]

    print(f"\nMODELO ESCOLHIDO: {best_model_name}")
    print(f"Performance (RMSE Validação): {best_run['rmse_val']:,.2f}")

    # 4. Gerar Previsões Finais (Para o Kaggle)
    print("\nA gerar previsões para o ficheiro de teste...")
    final_predictions = best_model_obj.predict(X_test_final)

    # 5. Criar DataFrame de Submissão
    submission_df = pd.DataFrame({
        'id': df_test_raw.index,  # Garante que os IDs correspondem ao ficheiro original
        'price': final_predictions
    })

    # 6. Guardar Resultados (Chama a função da Etapa 8)
    save_project_artifacts(submission_df, best_model_obj, best_run.to_dict())

    print("\nPROCESSO CONCLUÍDO COM SUCESSO!")

INICIAR O PROJETO...
Erro: Ficheiros 'train.csv' e 'test.csv' não encontrados.
A processar e transformar dados
Setup Concluído:
   - Treino: 150826 carros
   - Validação: 37707 carros
   - Features: 18 colunas

A otimizar hiperparâmetros para: XGBoost_Main...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
